# Lab 1: Prepare the Data

This lab fine-tunes **NVIDIA Nemotron 3 Nano 30B** on a contract review task using SageMaker AI serverless customization. You'll format the training data, register it in SageMaker, and set up the shared config all four notebooks use.

### Workshop notebooks

| # | Notebook | What you'll do |
|---|---|---|
| 1 | `1-prepare-data.ipynb` (this one) | Format ContractNLI for SFT and register the datasets in SageMaker |
| 2 | `2-fine-tune-llm.ipynb` | Launch the serverless LoRA fine-tuning job on Nemotron 3 Nano 30B |
| 3 | `3-evaluation.ipynb` | Score the base, fine-tuned, and frontier models |
| 4 | `4-deployment.ipynb` | Deploy the fine-tuned model to a SageMaker real-time endpoint |

Run them in order. Each notebook picks up where the previous one left off.

---

### The task

607 real NDAs, each checked against a fixed 17-point checklist. For each (contract, checklist item) pair the model must return a verdict and the span numbers that justify it.

<img src="./images/hypothesis_example.png" width="70%" alt="ContractNLI: contract spans with highlighted evidence and hypothesis verdicts">

Three verdicts are possible: **Entailment** (the contract supports the claim), **Contradiction** (the contract conflicts with it), or **NotMentioned** (the contract is silent). The output is strict JSON, one entry per checklist item:

```json
{"nda-2":  {"label": "Contradiction", "evidence": [3, 4]},
 "nda-5":  {"label": "Contradiction", "evidence": [3]},
 "nda-4":  {"label": "Entailment",    "evidence": [4]},
 "nda-11": {"label": "NotMentioned",  "evidence": []}}
```

A verdict without a citation is useless in practice. "Item 5 fails, see span 3" is verifiable in seconds; "Item 5 fails" is not. The evidence prediction is also what separates a trained model from one that guesses the most common label: always predicting `NotMentioned` scores 43% accuracy for free but zero on evidence.

### Why fine-tune instead of prompt?

On the 123-contract held-out set, the base Nemotron model scores **64.5 accuracy / 48.8 evidence-F1**. Claude Sonnet 5 zero-shot scores **83.6 / 67.1**. The fine-tuned model you'll train scores **87.4 / 75.4**, above the frontier model at roughly 8x lower cost per contract. Few-shot prompting makes the small model *worse*, not better. That's the signal this is a fine-tuning problem.

<img src="./images/model-comparison.png" width="50%" alt="Base, frontier and fine-tuned accuracy and evidence-F1">

### The data

[ContractNLI](https://stanfordnlp.github.io/contract-nli/) (Koreeda & Manning, *Findings of EMNLP 2021*), CC-BY-4.0. Splits are document-level: **423 train / 61 dev / 123 test**.


### Install requirements

In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


#### Setup and dependencies

Standard SageMaker boilerplate, no task-specific logic: it resolves the execution role,
the default bucket and the bucket prefix (`None` unless a SageMaker defaults config sets
one), and opens the boto3 clients. Of all that, this notebook reuses only `s3_client`,
`bucket_name` and `default_prefix`, all three in the upload cell; the role is what
notebook 2 passes to the training job. Notebooks 2 and 4 repeat this cell verbatim.

In [2]:
import boto3
from sagemaker.core.helper.session_helper import Session, get_execution_role

sess = Session()
sagemaker_session_bucket = None
if sagemaker_session_bucket is None and sess is not None:
    sagemaker_session_bucket = sess.default_bucket()

try:
    role = get_execution_role()
except ValueError:
    iam = boto3.client("iam")
    role = iam.get_role(RoleName="sagemaker_execution_role")["Role"]["Arn"]

s3_client = boto3.client("s3")
sess = Session(default_bucket=sagemaker_session_bucket)
sm_client = boto3.client("sagemaker", region_name=sess.boto_region_name)
bucket_name = sess.default_bucket()
default_prefix = sess.default_bucket_prefix

print(f"sagemaker role arn: {role}")
print(f"sagemaker bucket: {bucket_name}")
print(f"sagemaker session region: {sess.boto_region_name}")

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker role arn: arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRole-20201215T102238
sagemaker bucket: sagemaker-us-east-1-492681118881
sagemaker session region: us-east-1


### First: what is `contractnli.py`?

Every notebook in this lab starts with `import contractnli as C`. It's a small module
that sits next to these notebooks, and it holds only the mechanical parts, downloading
the data, reading it, and rendering the prompt. Anything that constitutes the *lesson*
(calling a model, scoring its answer) stays in the notebooks where you can see it.

You'll see these names used throughout. This is the whole surface:

| Call | Returns | Used for |
|---|---|---|
| `C.ensure_dataset("./data")` | the unpacked path | downloads the ContractNLI archive once |
| `C.load(split)` | `(documents, checklist)` | reads `train`, `dev` or `test` |
| `C.doc_spans(doc)` | `[(0, "text"), (1, "text"), ...]` | one contract as numbered clauses |
| `C.gold_for(doc)` | `{"nda-1": {"choice", "spans"}, ...}` | the expert answer for one contract |
| `C.build_prompt(doc, labels)` | one string | the whole request: instruction, contract, checklist |

One constant matters too: `C.INSTRUCTION`, the template `build_prompt` fills in. You
will print it in a moment.

`contractnli.py` also carries a two-turn `messages` variant of the same instruction
(`build_system`, `build_user`, `build_messages`). Nothing in *this* notebook uses it:
the training records below are single-string `prompt`/`completion` pairs built by
`build_prompt`. It exists for the callers that need chat turns: the frontier baseline in
notebook 3, which goes through the Bedrock Converse API, and the serving checks in notebook 4. The two formats carry the same instruction, but they're **not** the same string. The single-string form puts the contract before the
checklist, the two-turn form puts the checklist in the system turn, ahead of the
contract. Notebook 4 says why that is safe.

Open the file if you want, it's about 150 lines. The cells
below show what each helper returns.


### Download the dataset

`ensure_dataset()` fetches the CC-BY-4.0 archive from Stanford and unpacks it
into `./data`. Nothing else in the lab needs network access to the dataset.

`C.load(split)` then reads that split's JSON and returns `(documents, checklist)`.
Each document carries its text, the character offsets that cut it into numbered spans,
and the expert annotation. The checklist is byte-identical in all three files, same 17
items, same order, so only the train copy is kept, as `labels`, and the dev and test
copies go to `_`. That one dict renders the checklist into every prompt, in all three
splits.

In [3]:
import contractnli as C

C.ensure_dataset("./data")

train_docs, labels = C.load("train")
dev_docs, _ = C.load("dev")
test_docs, _ = C.load("test")

print(f"train {len(train_docs)} contracts | dev {len(dev_docs)} | test {len(test_docs)}")
print(f"checklist items: {len(labels)}")

train 423 contracts | dev 61 | test 123
checklist items: 17


#### What one document actually looks like

The raw shape you are working from. The
dataset gives you documents; the three helpers below are how you get from a document to
the pieces the prompt needs.


In [4]:
doc_example = train_docs[0]

print("One ContractNLI document is a plain dict. Its fields:\n")
for key, value in doc_example.items():
    size = f"{len(value):,} items" if isinstance(value, list) else f"{len(str(value)):,} chars"
    print(f"  doc[{key!r}]:{' ' * (20 - len(key))}{type(value).__name__:5s} {size}")

print("\nOnly three of those matter here, and `contractnli.py` has a helper for each.\n")

# 1. The contract, cut into the clauses the model will cite by number.
print("1. doc['spans'] holds (start, end) offsets into doc['text'], the dataset's own")
print("   clause split. C.doc_spans(doc) slices them out and numbers them:\n")
for number, text in C.doc_spans(doc_example)[:3]:
    print(f"     [{number}] {text[:62]}")

# 2. The expert answer. `choice` is the verdict, `spans` the clauses that justify it.
print("\n2. doc['annotation_sets'] holds the expert labels. C.gold_for(doc) unwraps it")
print("   to one entry per checklist item:\n")
gold_example = C.gold_for(doc_example)
for key in list(labels)[:2]:
    print(f"     {key}: {gold_example[key]}")

# 3. The checklist is the same for every contract. In this format it is rendered into
#    every prompt, which is why it comes from one dict rather than per-record text.
print("\n3. `labels`, the second value C.load() returned, is the checklist itself:\n")
first_key = list(labels)[0]
print(f"     labels[{first_key!r}]:")
for field, text in labels[first_key].items():
    print(f"       {field}: {text[:66]}")

One ContractNLI document is a plain dict. Its fields:

  doc['id']:                  int   2 chars
  doc['file_name']:           str   56 chars
  doc['text']:                str   8,585 chars
  doc['spans']:               list  65 items
  doc['annotation_sets']:     list  1 items
  doc['document_type']:       str   10 chars
  doc['url']:                 str   73 chars

Only three of those matter here, and `contractnli.py` has a helper for each.

1. doc['spans'] holds (start, end) offsets into doc['text'], the dataset's own
   clause split. C.doc_spans(doc) slices them out and numbers them:

     [0] NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT
     [1] This NON-DISCLOSURE AND CONFIDENTIALITY AGREEMENT (“Agreement”
     [2] (i) the Office of the United Nations High Commissioner for Ref

2. doc['annotation_sets'] holds the expert labels. C.gold_for(doc) unwraps it
   to one entry per checklist item:

     nda-11: {'choice': 'NotMentioned', 'spans': []}
     nda-16: {'choice': 'Entailment

### The checklist the model has to answer

The 17 hypotheses, each with the short description that names it. Not background
reading: `build_prompt()` renders these same two fields, one line per item, into the
`CHECKLIST:` block of the prompt, 2,324 characters of the 3,234 every prompt carries
besides the contract itself. `labels` is
loaded once, from the train file, and the same dict builds all three splits, so every
contract is judged against this list.

The `int(...)` sort key is for readability only: the dataset's order starts at
`nda-11`, and plain sorting would put `nda-10` before `nda-2`. Numbers 6, 9 and 14 are
unused, which is why 17 items reach `nda-20`. The prompt and the completion both
keep the dataset's order, that is why the JSON in the introduction starts at `nda-11`.

In [5]:
for k, v in sorted(labels.items(), key=lambda kv: int(kv[0].split("-")[1])):
    print(f"{k:7s} [{v['short_description']}]")
    print(f"        {v['hypothesis']}")

nda-1   [Explicit identification]
        All Confidential Information shall be expressly identified by the Disclosing Party.
nda-2   [None-inclusion of non-technical information]
        Confidential Information shall only include technical information.
nda-3   [Inclusion of verbally conveyed information]
        Confidential Information may include verbally conveyed information.
nda-4   [Limited use]
        Receiving Party shall not use any Confidential Information for any purpose other than the purposes stated in Agreement.
nda-5   [Sharing with employees]
        Receiving Party may share some Confidential Information with some of Receiving Party's employees.
nda-7   [Sharing with third-parties]
        Receiving Party may share some Confidential Information with some third-parties (including consultants, agents and professional advisors).
nda-8   [Notice on compelled disclosure]
        Receiving Party shall notify Disclosing Party in case Receiving Party is required by law, regu

### Look at one real contract

Below is one of the shortest NDAs in the test set, split into the numbered spans
the model will see, followed by the gold answer. Read span [3] and then look at
the labels for `nda-5` (sharing with employees) and `nda-7` (sharing with
third-parties).

In [6]:
doc = sorted(test_docs, key=lambda d: len(d["text"]))[1]
spans = C.doc_spans(doc)

print(f"{doc['file_name']}  |  {len(doc['text'].split())} words, {len(spans)} spans\n")
for i, t in spans:
    print(f"[{i}] {t[:160]}")

1023734_0000912057-96-023266_document_16.txt  |  232 words, 13 spans

[0] NAVIDEC, INCORPORATED
[1] TRADE SECRET/NON-DISCLOSURE AGREEMENT
[2] In consideration of the mutual promises made herein, as well as the agreement between Navidec, Incorporated and _______________ , the parties hereby agree as fo
[3] ____________________ , agrees that, in consideration for being shown or told about certain trade secrets or property belonging to Navidec, Incorporated, _______
[4] Further, ___________________ , agrees not to use, either directly or indirectly any of the material, ideas, objects or portions thereof of said trade secret or 
[5] Any dispute that arises hereunder shall be resolved by arbitration pursuant to the rules of the American Arbitration Association or the rules of the State of Co
[6] In the event that any litigation or arbitration is commenced to enforce any of the provisions of this agreement, the prevailing party of said litigation shall b
[7] This agreement shall be governed 

The gold answer for the same contract, one line per checklist item. `C.gold_for()`
returns the `annotations` of the document's single annotation set: per item, a `choice`
from the three labels and `spans`, the span numbers the annotator pointed to, in the
same numbering as the `[i]` markers above. `spans` is non-empty for exactly the
`Entailment` and `Contradiction` entries, in all 10,319 judgements across the three
splits, so the citation rule the prompt states is one the data already obeys.

`gold_json()` further down turns this into the completion, renaming `choice` to
`label` and `spans` to `evidence`. It emits the items in the checklist's order rather
than the numeric order sorted here, which is why the JSON in the introduction starts at
`nda-11`.

In [7]:
gold = C.gold_for(doc)
print("gold answer:\n")
for k in sorted(gold, key=lambda x: int(x.split("-")[1])):
    v = gold[k]
    ev = f"  evidence={v['spans']}" if v["spans"] else ""
    print(f"{k:7s} {v['choice']:14s}{ev}   [{labels[k]['short_description']}]")

gold answer:

nda-1   NotMentioned     [Explicit identification]
nda-2   Contradiction   evidence=[3, 4]   [None-inclusion of non-technical information]
nda-3   NotMentioned     [Inclusion of verbally conveyed information]
nda-4   Entailment      evidence=[4]   [Limited use]
nda-5   Contradiction   evidence=[3]   [Sharing with employees]
nda-7   Contradiction   evidence=[3]   [Sharing with third-parties]
nda-8   NotMentioned     [Notice on compelled disclosure]
nda-10  NotMentioned     [Confidentiality of Agreement]
nda-11  NotMentioned     [No reverse engineering]
nda-12  NotMentioned     [Permissible development of similar information]
nda-13  NotMentioned     [Permissible acquirement of similar information]
nda-15  NotMentioned     [No licensing]
nda-16  NotMentioned     [Return of confidential information]
nda-17  NotMentioned     [Permissible copy]
nda-18  NotMentioned     [No solicitation]
nda-19  NotMentioned     [Survival of obligations]
nda-20  NotMentioned     [Permissible po

### Check the answer against the clause you just read

This is the contract from the introduction, and span [3] is that blanket
prohibition. Notice it drives three separate `Contradiction` verdicts:

| Item | Subject | Evidence |
|---|---|---|
| `nda-5` | sharing with employees | `[3]` |
| `nda-7` | sharing with third parties | `[3]` |
| `nda-2` | only technical information is confidential | `[3, 4]` |

One clause, three verdicts, which is why the model has to reason over the whole
document per item rather than retrieve one passage per question.

Notice also that 13 of the 17 items are `NotMentioned`. Short NDAs are silent on most of
the checklist, and that skew is why accuracy alone is a weak metric here: over the whole
test split 43% of decisions are `NotMentioned`, so always answering it scores 43% without
reading anything, see notebook 3.


### The prompt

Everything the model will ever see is one string: the template below with three
slots filled in, the number of checklist items, the contract as numbered spans, and the
checklist itself.

**The format: chat completion.** Serverless customization accepts a record as a
`prompt`/`completion` pair, which is the shape this notebook writes. Everything the
model reads goes in `prompt`, and the single thing it must produce goes in `completion`;
there are no roles and no turn boundaries to get right. The order inside the prompt is
instruction, then contract, then checklist, then the required output shape, the
checklist sits *after* the contract so the last thing the model reads before answering
is what it is being asked.

It's defined **once**, in `contractnli.py`, and used by every notebook, data prep
here, the frontier baseline in notebook 3, and the serving checks in notebook 4. That's deliberate: the promise that *the training prompt is byte-identical to the inference
prompt* only holds if there is exactly one copy. A second copy pasted into a notebook is
how train/serve skew gets introduced.


In [8]:
# The exact template. {n}, {spans} and {checklist} are the only substitutions.
# build_prompt() can append C.NO_THINK, but this lab passes no_think=False
# everywhere: Nemotron ignores the directive, so the dataset carries the fix.
print("=" * 70, "\nINSTRUCTION\n", "=" * 70, sep="")
print(C.INSTRUCTION)


INSTRUCTION
You are a contract review assistant. You review a non-disclosure agreement (NDA) against a fixed checklist of {n} legal hypotheses.

For EACH hypothesis, decide:
- "Entailment": the contract states or implies the hypothesis is true.
- "Contradiction": the contract states something that conflicts with the hypothesis.
- "NotMentioned": the contract does not address it.

Also cite the span numbers that justify the decision (the exact spans a lawyer would point to). Cite spans only for Entailment or Contradiction; use an empty list for NotMentioned. Read exceptions and carve-outs carefully: a clause with an exception may contradict a hypothesis stated absolutely.

CONTRACT (numbered spans):
{spans}

CHECKLIST:
{checklist}

Respond with JSON only, no other text:
{{"nda-1": {{"label": "Entailment|Contradiction|NotMentioned", "evidence": [span numbers]}}, ...}}
Include an entry for every hypothesis key listed above.


To experiment with the wording, set `C.INSTRUCTION` here and re-run the record build
below, every notebook then picks up your version:

```python
C.INSTRUCTION = """...your wording, keeping {n}, {spans} and {checklist}..."""
```

#### Why completions start with `<think>\n</think>`

Nemotron 3 Nano is a reasoning model, and that creates a real problem out of the box. Its chat template automatically opens a `<think>` block at the start of every assistant turn, but nothing closes it. On a 17-item checklist it fills the whole
generation budget reasoning and never reaches the JSON. Every metric then reads 0.

Qwen3 has the same tendency but obeys `/no_think` in the prompt. Nemotron ignores it,
whether it arrives as a system field, in the user turn, or both, so the switch has to go
somewhere the model reads as its own output. Every training completion below therefore
begins with `<think>\n</think>\n`: the model learns to close the block the template
opened, then answer. At inference it emits `</think>` and goes straight into the JSON.

`/no_think` is not used anywhere in this lab, so both builders pass `no_think=False` and
train, validation and test prompts are the same string. See
[`nemotron_support.py`](nemotron_support.py) for the measured numbers and for the
one-line template edit that reaches the same result without touching the dataset.


### Build the training records

One training record = one contract, with all 17 verdicts in the completion. So
423 records, but each carries 17 supervised decisions plus the evidence spans,
which is roughly 7,200 labelled judgements.

**The format: `prompt`/`completion`.** Two string fields per line, no roles:

```json
{"prompt": "You are a contract review assistant... CONTRACT (numbered spans):\n[0] ... CHECKLIST: ...",
 "completion": "<think>\n</think>\n{\"nda-11\": {\"label\": \"NotMentioned\", ...}}"}
```

The recipes accept this alongside the role-tagged `messages` shape, and for a
single-turn task it is the simpler of the two: there's exactly one boundary in the
record, and it is the one the trainer needs. The completion is the only place this
dataset differs from a non-reasoning model's: the empty block, then the answer.

#### Why the test split looks different

Train and validation use `prompt`/`completion`. The **test** split keeps
`query`/`response`, because notebook 3 scores it with `CustomScorerEvaluator`, which
pins the evaluation task to `gen_qa`, a format whose fields are exactly `query`,
`response` and an optional `system`.

The reference answer stays plain JSON: the reasoning block belongs to the model's turn,
not to the gold answer.

```json
{"query":    "You are a contract review assistant... CONTRACT ... CHECKLIST: ...",
 "response": "{\"nda-11\": ...}"}
```

The optional `system` field is unused. An earlier version of this dataset set it to
`/no_think`; removing it scored marginally higher.


All three splits at once. `gold_json` renames the annotator's fields, `choice` to
`label` and `spans` to `evidence`. Both builders call the same `build_prompt()`, so the
test split is that string under genqa's names rather than a second prompt.

In [9]:
import json

import nemotron_support as N

label_keys = list(labels.keys())          # the checklist's own order


def gold_json(doc):
    """The expert answer for one contract, as the exact JSON the model must emit.
    Only the field names change: `choice` becomes `label`, `spans` becomes `evidence`.
    """
    g = C.gold_for(doc)
    return json.dumps({k: {"label": g[k]["choice"], "evidence": list(g[k]["spans"])}
                       for k in label_keys if k in g})


def make_records(docs):
    """Training records: one prompt/completion pair per contract. `training_record`
    prefixes the completion with the empty reasoning block."""
    return [N.training_record(C.build_prompt(d, labels, no_think=False), gold_json(d))
            for d in docs]


def make_test_records(docs):
    """Evaluation records. The managed scorer reads `query`/`response` (genqa) and
    nothing else, so the test split keeps that shape, see the note below. The
    reference answer is plain JSON, with no block."""
    return [N.eval_record(C.build_prompt(d, labels, no_think=False), gold_json(d))
            for d in docs]


records = {"train": make_records(train_docs),
           "val": make_records(dev_docs),
           "test": make_test_records(test_docs)}

for name, rows in records.items():
    field = "query" if name == "test" else "prompt"
    avg = sum(len(r[field]) for r in rows) // len(rows)   # chars, not tokens
    print(f"{name:5s}: {len(rows):4d} records, avg prompt {avg:6d} chars "
          f"(~{avg // 4} tokens)")


train:  423 records, avg prompt  14650 chars (~3662 tokens)
val  :   61 records, avg prompt  15730 chars (~3932 tokens)
test :  123 records, avg prompt  14843 chars (~3710 tokens)


One complete training record, the prompt the model reads and the JSON it must
produce. The cell picks the train record with the **shortest** prompt: 4,783 characters,
of which the contract is only 1,560 over 18 spans, against a median prompt of ~13,554,
so one whole contract fits on screen. It's the smallest NDA in the split, not a typical
one. The completion is stored as a single 912-character line, the 17-character reasoning
block plus 895 of JSON. `indent=1` reformats it for reading here and `[:800]` trims the
display, not the record: all 17 verdicts are in the target.


In [13]:
  sample = min(records["train"], key=lambda r: len(r["prompt"]))

  # The completion is the empty reasoning block followed by the JSON, so the block
  # comes off before parsing and goes back on for display.
  answer = sample["completion"].removeprefix(N.EMPTY_REASONING)

  print("=" * 70, "\nPROMPT\n", "=" * 70, sep="")
  print(sample["prompt"])
  print("\n" + "=" * 70, "\nCOMPLETION\n", "=" * 70, sep="")
  print(N.EMPTY_REASONING + json.dumps(json.loads(answer), indent=1)[:800], "...")

PROMPT
You are a contract review assistant. You review a non-disclosure agreement (NDA) against a fixed checklist of 17 legal hypotheses.

For EACH hypothesis, decide:
- "Entailment": the contract states or implies the hypothesis is true.
- "Contradiction": the contract states something that conflicts with the hypothesis.
- "NotMentioned": the contract does not address it.

Also cite the span numbers that justify the decision (the exact spans a lawyer would point to). Cite spans only for Entailment or Contradiction; use an empty list for NotMentioned. Read exceptions and carve-outs carefully: a clause with an exception may contradict a hypothesis stated absolutely.

CONTRACT (numbered spans):
[0] Department of State
[1] Washington, DC 20520
[2] NON-DISCLOSURE AGREEMENT
[3] By signing below I agree to the following conditions:
[4] 1) I will hold confidential the content of the Foreign Service Oral Assessment.
[5] 2) I will not disclose, publish, reproduce or transmit any examination mat

#### Write to disk and upload to Amazon S3

`shutil.rmtree` removes `./sft_data` first, so a re-run cannot leave a stale file
behind. Each split is then written as `./sft_data/<split>/dataset.jsonl`, one JSON
object per line, the shape `DataSet.create` validates before registering it below. The
directory names are the keys of `records`, so the `dev` documents are written as `val`,
and that is the name notebook 2 fetches the validation set by. The train file comes to
6.8 MB over its 423 lines, about 16 KB a record.

The key joins `DATASET_PREFIX` (`contractnli-nda-review`, from `config.py`) to
`default_prefix`, which the setup cell read from `sess.default_bucket_prefix`, a
leading key segment when a SageMaker config sets one, and `None` otherwise, which is why
the key is assembled with a conditional. Each file lands at
`s3://<bucket>/[<prefix>/]datasets/contractnli-nda-review/<split>/dataset.jsonl`, and
the three URIs accumulate in `s3_paths`, the dict the next cell registers.

In [14]:
import pathlib
import shutil

from config import DATASET_PREFIX

local = pathlib.Path("./sft_data")
if local.exists():
    shutil.rmtree(local)

for name, rows in records.items():
    d = local / name
    d.mkdir(parents=True, exist_ok=True)
    with open(d / "dataset.jsonl", "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")

input_path = (f"{default_prefix}/datasets/{DATASET_PREFIX}" if default_prefix
              else f"datasets/{DATASET_PREFIX}")

s3_paths = {}
for name in records:
    key = f"{input_path}/{name}/dataset.jsonl"
    s3_client.upload_file(str(local / name / "dataset.jsonl"), bucket_name, key)
    s3_paths[name] = f"s3://{bucket_name}/{key}"
    print(s3_paths[name])

s3://sagemaker-us-east-1-492681118881/datasets/contractnli-nda-review-nothink/train/dataset.jsonl
s3://sagemaker-us-east-1-492681118881/datasets/contractnli-nda-review-nothink/val/dataset.jsonl
s3://sagemaker-us-east-1-492681118881/datasets/contractnli-nda-review-nothink/test/dataset.jsonl


#### Register the datasets

`DataSet.create` writes a registry entry pointing at the S3 object you just uploaded,
and what that buys you is a name. Notebook 2 calls
`DataSet.get(name=f"{DATASET_PREFIX}-train")` and hands the returned object to
`SFTTrainer(training_dataset=...)`, which resolves it to the entry's name and version
rather than an S3 URI, so the job records which dataset it consumed. Both evaluators in
notebook 3 resolve the test entry the same way. `wait=True` blocks until each import
reaches `Available`, and raises if it reaches `ImportFailed` instead.

| dataset | technique | consumed by |
|---|---|---|
| `contractnli-nda-review-train` | `SFT` | notebook 2, `training_dataset=` |
| `contractnli-nda-review-val` | `SFT` | notebook 2, `validation_dataset=` |
| `contractnli-nda-review-test` | none | notebook 3, `dataset=` |

The technique is stored as the search keyword `customization_technique:sft`, a label to
find the dataset by, not a constraint; nothing reads it back at training time, since the
trainer knows its own technique. The enum offers `SFT`, `DPO` and `RLVR` and nothing for
evaluation data, so the test split is registered without one.

`create` also downloads each file and matches it against the formats the registry
knows (the prompt/completion shape for train and val, `genqa` for the test split) so
the wrong shape fails here rather than inside a training job. It reads only the file's first
record.
The match is pass/fail and no format name is stored anywhere in the entry, which is why
the registry console shows an empty **Format** column for every dataset, it is not a
sign that anything is wrong with yours.

> **Note:** `create` overwrites nothing, it reads the current version
> and imports the next major one, so a second run leaves `2.0.0` beside `1.0.0`, and a
> lookup by bare name returns the latest. But the entry records only a bucket and a key,
> with no version id or checksum, and the cell above always writes the same key. Every
> version therefore resolves to the *same* `dataset.jsonl`: the version list is a
> history of registrations, not of your data. Re-upload before you re-register, and
> never expect `1.0.0` to still hold the records it was created with.

In [15]:
from sagemaker.ai_registry.dataset import DataSet
from sagemaker.ai_registry.dataset_utils import CustomizationTechnique


def register(name, source, technique=None):
    kwargs = dict(name=name, source=source, wait=True)
    if technique is not None:
        kwargs["customization_technique"] = technique
    ds = DataSet.create(**kwargs)
    print(f"created dataset: {name}")
    return ds


training_dataset = register(f"{DATASET_PREFIX}-train", s3_paths["train"], CustomizationTechnique.SFT)
val_dataset = register(f"{DATASET_PREFIX}-val", s3_paths["val"], CustomizationTechnique.SFT)
test_dataset = register(f"{DATASET_PREFIX}-test", s3_paths["test"])

[09/10/26 01:58:52] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=8373;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=461386;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#322\322]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

                    INFO     Role not provided. Using validated caller role:                         ]8;id=845421;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=855895;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRo               
                             le-20201215T102238                                                                    

Output()

Final Resource Status: Available

created dataset: contractnli-nda-review-nothink-train


...

created dataset: contractnli-nda-review-nothink-val


[09/10/26 01:58:57] INFO     Role not provided. Using validated caller role:                         ]8;id=275083;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py\defaults.py]8;;\:]8;id=3153;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/defaults.py#90\90]8;;\
                             arn:aws:iam::492681118881:role/service-role/AmazonSageMaker-ExecutionRo               
                             le-20201215T102238                                                                    

Output()

Final Resource Status: Available

created dataset: contractnli-nda-review-nothink-test


### What you built

`contractnli-nda-review-train` (423 records), `-val` (61) and `-test` (123 held out),
registered and ready. Notebooks 2 and 3 look them up by name, no variable defined in
this notebook has to survive.

Continue to **notebook 2** to run the serverless LoRA fine-tuning job.